<a href="https://colab.research.google.com/github/Abrish-21/looped_model_experments/blob/main/looped_models_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recurrent depth: an inference study + an optimizer pilot
**Part A** – Huginn-0125 (Geiping et al. 2025): how does accuracy and latent convergence change with `num_steps`?
**Part B** – tiny weight-tied (looped) transformer: AdamW vs Muon, learning-rate stability.



*Hypotheses (write yours before running):*
- A: steps needed for a given accuracy grow with problem difficulty; the output distribution stops changing after some number of steps.
- B: the range of learning rates that trains the looped model stably differs between AdamW and Muon.

## Part A: Huginn inference study

In [ ]:
!pip install -q "transformers==4.49.0" accelerate

In [ ]:
import transformers
print(transformers.__version__)   # should print 4.49.0

4.49.0


In [ ]:
import torch, random, re, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
name = "tomg-group-umd/huginn-0125"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, trust_remote_code=True).to("cuda").eval()
print(torch.cuda.memory_allocated()/1e9, "GB")
# If bf16 is very slow/garbled on the T4, try torch.float16 and re-check outputs. If cache errors appear, pin an older transformers.

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/178 [00:00<?, ?B/s]

14.270731264 GB


In [ ]:
print(next(model.parameters()).dtype)

torch.float32


In [ ]:
model = model.to(torch.bfloat16)
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated()/1e9, "GB")   # should drop to about 7

7.141408256 GB


In [ ]:
def chat_ids(q):
    s = tok.apply_chat_template([{"role":"user","content":q}], tokenize=False, add_generation_prompt=True)
    return tok.encode(s, return_tensors="pt", add_special_tokens=False)

@torch.no_grad()
def logits_for(ids, steps):
    out = model(ids.to("cuda"), num_steps=steps)
    return out.logits            # check shape: [1, L, vocab]

# sanity check: one generation + shapes
cfg = GenerationConfig(max_new_tokens=24, stop_strings=["<|end_text|>","<|end_turn|>"], use_cache=True, do_sample=False, temperature=None, top_k=None)
ids = chat_ids("What is 4821 + 7359? Answer with just the number.").to("cuda")
t=time.time(); o = model.generate(ids, cfg, num_steps=16, tokenizer=tok)
seq = o.sequences if hasattr(o,"sequences") else o
print(tok.decode(seq[0][ids.shape[1]:], skip_special_tokens=True), f"({time.time()-t:.1f}s)")
print(logits_for(ids, 8).shape)

4821 + 7359 = 12170 (10.5s)
torch.Size([1, 30, 65536])


In [ ]:
# Synthetic task with controllable difficulty: n-digit addition
def make_problem(n, rng):
    a, b = rng.randint(10**(n-1), 10**n-1), rng.randint(10**(n-1), 10**n-1)
    return f"What is {a} + {b}? Answer with just the number.", str(a+b)

STEPS = [2, 4, 8, 16, 32, 64]
DIGITS = [2, 4, 6]
N_PER_CELL = 100          # lower if slow; forward cost grows linearly with steps

@torch.no_grad()
def teacher_forced(q, ans, steps):
    x, y = re.findall(r"\d+", q)                       # the two operands
    prefix = tok.encode(f"{x} + {y} =", add_special_tokens=False, return_tensors="pt")
    p = torch.cat([chat_ids(q), prefix], 1)
    a = tok.encode(" " + ans, add_special_tokens=False, return_tensors="pt")
    ids = torch.cat([p, a], 1)
    lg = logits_for(ids, steps)[0]
    L = p.shape[1]
    pred = lg[L-1:-1].argmax(-1).cpu()
    lp = torch.log_softmax(lg[L-1:-1].float(), -1).cpu()
    ll = lp[torch.arange(a.shape[1]), a[0]].mean().item()
    return bool((pred == a[0]).all()), ll

rng = random.Random(0)
problems = {n: [make_problem(n, rng) for _ in range(N_PER_CELL)] for n in DIGITS}
rows = []
for n in DIGITS:
    for s in STEPS:
        t = time.time()
        for q, ans in problems[n]:
            ok, ll = teacher_forced(q, ans, s)
            rows.append(dict(digits=n, steps=s, ok=ok, ll=ll))
        pd.DataFrame(rows).to_csv("huginn_addition.csv", index=False)   # save as you go
        d = pd.DataFrame(rows); d = d[(d.digits==n)&(d.steps==s)]
        print(n, s, f"acc={d.ok.mean():.2f}  ll={d.ll.mean():.2f}  {time.time()-t:.0f}s")

2 2 acc=0.00  ll=-2.87  28s
2 4 acc=0.20  ll=-1.27  47s
2 8 acc=0.78  ll=-0.31  84s
2 16 acc=0.93  ll=-0.09  158s
2 32 acc=0.94  ll=-0.07  306s
2 64 acc=0.94  ll=-0.07  600s
4 2 acc=0.00  ll=-3.00  30s
4 4 acc=0.00  ll=-1.89  49s
4 8 acc=0.26  ll=-0.68  88s
4 16 acc=0.69  ll=-0.23  164s
4 32 acc=0.70  ll=-0.23  315s
4 64 acc=0.70  ll=-0.23  620s
6 2 acc=0.00  ll=-2.72  32s
6 4 acc=0.00  ll=-2.17  51s


In [ ]:
df = pd.DataFrame(rows)
def ci(x, B=1000):
    x = np.asarray(x, float); bs = [np.random.choice(x, len(x)).mean() for _ in range(B)]
    return np.percentile(bs, [2.5, 97.5])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n in DIGITS:
    g = df[df.digits==n].groupby("steps")
    m = g.ok.mean(); c = np.array([ci(v.ok) for _, v in g])
    ax[0].plot(m.index, m.values, marker="o", label=f"{n}-digit"); ax[0].fill_between(m.index, c[:,0], c[:,1], alpha=.2)
    ax[1].plot(g.ll.mean().index, g.ll.mean().values, marker="o", label=f"{n}-digit")
for a_, t_ in zip(ax, ["Teacher-forced exact match", "Mean answer log-prob"]):
    a_.set_xscale("log", base=2); a_.set_xlabel("recurrent steps"); a_.set_title(t_); a_.legend()
plt.tight_layout(); plt.savefig("fig_A_accuracy_vs_steps.png", dpi=150)

In [ ]:
# Experiment 2: does the next-token distribution settle?  KL( p_k || p_64 ) on the first answer token
KS = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48]
def kl_to_ref(q, ref=64):
    p = chat_ids(q)
    pr = torch.log_softmax(logits_for(p, ref)[0, -1].float(), -1)
    out = []
    for k in KS:
        pk = torch.log_softmax(logits_for(p, k)[0, -1].float(), -1)
        out.append((pk.exp() * (pk - pr)).sum().item())
    return out
res = {n: np.array([kl_to_ref(q) for q, _ in problems[n][:30]]) for n in DIGITS}
plt.figure(figsize=(6,4))
for n in DIGITS:
    plt.plot(KS, np.median(res[n], 0), marker="o", label=f"{n}-digit (median)")
plt.yscale("log"); plt.xscale("log", base=2); plt.xlabel("recurrent steps k"); plt.ylabel("KL(p_k || p_64)"); plt.legend()
plt.title("Distance of k-step prediction to 64-step prediction"); plt.tight_layout(); plt.savefig("fig_A_kl.png", dpi=150)

## Part B: tiny looped transformer, AdamW vs Muon
Task: sort 10 digits (non-autoregressive). One transformer block is reused T=8 times with input injection. Muon is applied to hidden 2-D matrices; embeddings, norms and head use AdamW.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import math, json, time
import torch, torch.nn as nn, torch.nn.functional as F

dev = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- task: sort a list of digits (non-autoregressive) ----------------
VOCAB, N = 16, 10
def batch(bs, g):
    x = torch.randint(0, VOCAB, (bs, N), generator=g)
    y, _ = torch.sort(x, dim=1)
    return x.to(dev), y.to(dev)

# ---------------- looped model: one shared block applied T times ----------------
class Block(nn.Module):
    def __init__(s, d, h):
        super().__init__()
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.qkv, s.o = nn.Linear(d, 3 * d, bias=False), nn.Linear(d, d, bias=False)
        s.up, s.down = nn.Linear(d, 4 * d, bias=False), nn.Linear(4 * d, d, bias=False)
        s.h = h
    def forward(s, x):
        B, L, D = x.shape
        q, k, v = s.qkv(s.n1(x)).view(B, L, 3, s.h, D // s.h).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v)            # bidirectional
        x = x + s.o(a.transpose(1, 2).reshape(B, L, D))
        return x + s.down(F.gelu(s.up(s.n2(x))))

class Looped(nn.Module):
    def __init__(s, d=128, h=4, T=8):
        super().__init__()
        s.emb, s.pos = nn.Embedding(VOCAB, d), nn.Embedding(N, d)
        s.inject = nn.Linear(2 * d, d, bias=False)             # input injection each loop
        s.block, s.norm, s.head = Block(d, h), nn.LayerNorm(d), nn.Linear(d, VOCAB, bias=False)
        s.T = T
    def forward(s, x, T=None):
        e = s.emb(x) + s.pos(torch.arange(N, device=x.device))
        z = torch.zeros_like(e)
        for _ in range(T or s.T):
            z = s.block(s.inject(torch.cat([z, e], -1)))
        return s.head(s.norm(z))

# ---------------- Muon (orthogonalised momentum) for 2-D hidden weights ----------------
def newton_schulz(G, steps=5):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    X = X / (X.norm() + 1e-7)
    tr = X.shape[0] > X.shape[1]
    if tr: X = X.T
    for _ in range(steps):
        A = X @ X.T
        X = a * X + (b * A + c * A @ A) @ X
    return X.T if tr else X

class Muon(torch.optim.Optimizer):
    def __init__(s, params, lr=0.02, momentum=0.95):
        super().__init__(params, dict(lr=lr, momentum=momentum))
    @torch.no_grad()
    def step(s):
        for g in s.param_groups:
            for p in g["params"]:
                if p.grad is None: continue
                st = s.state[p]
                buf = st.setdefault("buf", torch.zeros_like(p))
                buf.mul_(g["momentum"]).add_(p.grad)
                upd = p.grad.add(buf, alpha=g["momentum"])      # nesterov
                upd = newton_schulz(upd).to(p.dtype)
                upd *= max(1, p.shape[0] / p.shape[1]) ** 0.5
                p.add_(upd, alpha=-g["lr"])

def make_opt(name, model, lr):
    mats = [p for n, p in model.named_parameters() if p.ndim == 2 and "emb" not in n and "pos" not in n and "head" not in n]
    rest = [p for n, p in model.named_parameters() if not any(p is m for m in mats)]
    if name == "adamw":
        return [torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)]
    # Muon on hidden matrices, AdamW (small fixed lr) on the rest
    return [Muon(mats, lr=lr), torch.optim.AdamW(rest, lr=1e-3, weight_decay=0.01)]

# ---------------- train one run ----------------
def run(opt_name, lr, seed, steps=1500, bs=128, T=8, d=128, log_every=25):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed)
    model = Looped(d=d, T=T).to(dev)
    opts = make_opt(opt_name, model, lr)
    warm = 100
    hist = dict(step=[], loss=[], gnorm=[])
    diverged = False
    for it in range(steps):
        scale = min(1.0, (it + 1) / warm) * 0.5 * (1 + math.cos(math.pi * it / steps))
        for o in opts:
            for gr in o.param_groups:
                gr.setdefault("base", gr["lr"]); gr["lr"] = gr["base"] * scale
        x, y = batch(bs, g)
        loss = F.cross_entropy(model(x).reshape(-1, VOCAB), y.reshape(-1))
        if not torch.isfinite(loss): diverged = True; break
        for o in opts: o.zero_grad(set_to_none=True)
        loss.backward()
        gn = torch.sqrt(sum((p.grad.float() ** 2).sum() for p in model.parameters() if p.grad is not None)).item()
        for o in opts: o.step()
        if it % log_every == 0:
            hist["step"].append(it); hist["loss"].append(loss.item()); hist["gnorm"].append(gn)
    # evaluate exact-match accuracy on fresh data (same loop depth as training)
    model.eval(); correct = tot = 0
    ge = torch.Generator().manual_seed(10_000 + seed)
    with torch.no_grad():
        for _ in range(8):
            x, y = batch(256, ge)
            correct += (model(x).argmax(-1) == y).all(1).sum().item(); tot += len(x)
    acc = 0.0 if diverged else correct / tot
    spikes = sum(1 for a, b in zip(hist["loss"], hist["loss"][1:]) if b > 2 * a and a < 1.5)
    return dict(opt=opt_name, lr=lr, seed=seed, T=T, acc=acc, diverged=diverged,
                final_loss=hist["loss"][-1] if hist["loss"] else float("nan"),
                spikes=spikes, hist=hist)



In [ ]:
import itertools, json
LRS = {"adamw": [3e-4, 1e-3, 3e-3, 1e-2, 3e-2], "muon": [3e-3, 1e-2, 3e-2, 1e-1, 3e-1]}
SEEDS = [0, 1, 2]
results = []
t0 = time.time()
for opt in LRS:
    for lr in LRS[opt]:
        for sd in SEEDS:
            r = run(opt, lr, sd, steps=1500, bs=128, T=8, d=128)
            results.append(r)
            print(f"{opt:5s} lr={lr:<7g} seed={sd} acc={r['acc']:.3f} loss={r['final_loss']:.3f} spikes={r['spikes']} div={r['diverged']}  [{time.time()-t0:.0f}s]")
        json.dump(results, open("looped_opt_results.json", "w"))

In [ ]:
rs = pd.DataFrame([{k: v for k, v in r.items() if k != "hist"} for r in results])
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for opt, c in [("adamw", "C0"), ("muon", "C1")]:
    s = rs[rs.opt == opt].groupby("lr").acc
    ax[0].errorbar(s.mean().index, s.mean().values, yerr=s.std().values, marker="o", color=c, label=opt, capsize=3)
ax[0].set_xscale("log"); ax[0].set_xlabel("learning rate"); ax[0].set_ylabel("exact-match accuracy"); ax[0].legend(); ax[0].set_title("LR sensitivity (mean±sd, 3 seeds)")
best = {o: rs[rs.opt == o].groupby("lr").acc.mean().idxmax() for o in LRS}
for r in results:
    if r["seed"] == 0 and r["lr"] == best[r["opt"]]:
        ax[1].plot(r["hist"]["step"], r["hist"]["loss"], label=f"{r['opt']} lr={r['lr']}")
        ax[2].plot(r["hist"]["step"], r["hist"]["gnorm"], label=f"{r['opt']} lr={r['lr']}")
ax[1].set_yscale("log"); ax[1].set_title("Training loss (best LR each)"); ax[1].legend()
ax[2].set_yscale("log"); ax[2].set_title("Gradient norm (best LR each)"); ax[2].legend()
plt.tight_layout(); plt.savefig("fig_B_optimizers.png", dpi=150)
print(rs.groupby(["opt", "lr"]).agg(acc=("acc", "mean"), diverged=("diverged", "sum"), spikes=("spikes", "mean")))

## Optional extensions (only if time)
- Vary loop count T in {2, 4, 8, 16} and check whether Muon's advantage (or disadvantage) changes with depth.
- Add a non-looped baseline with the same compute (T separate blocks).
- Evaluate a trained model at T_test > T_train to see whether extra loops help or hurt.

## Write-up template (keep it to one page)
1. Question and hypothesis. 2. Setup (model, tasks, steps, seeds, versions). 3. Figures A and B. 4. What the data show, and what they do not. 5. Limitations (small scale, one model, synthetic tasks, few seeds). 6. One concrete follow-up question about optimizers for looped models.